In [1]:
%%capture
# 1. Instalamos geoai (al ser muy ligero, dejamos que lo baje de internet)
%pip install geoai-py -q


In [2]:
import os
import geoai

In [3]:
import os

input_folder = "/kaggle/input/datasets/davidenriquealba/parcela-b7"
out_folder = "/kaggle/working/maskrcnn"
os.makedirs(out_folder, exist_ok=True)

Entrenamos el modelo Mask R-CNN sobre nuestros tiles generados, para que clasifique, localice bboxes y aplique máscaras a cada cepa detectada.

In [4]:
geoai.train_instance_segmentation_model(
    images_dir=f"{input_folder}/images",
    labels_dir=f"{input_folder}/labels",
    output_dir=f"{out_folder}",
    num_classes=2,  # clase fondo y clase cepa. En un futuro, se añadirá clase tronco
    num_channels=5, # 3 para imágenes RGB, 5 para imágenes MSP
    batch_size=4, # Para no consumir excesiva VRAM, y no provocar un error de Out of Memory a mitad de ejecución.
    num_epochs=10, # 10 para una PoC, 50 para entrenamiento real con dataset augmentado.
    learning_rate=0.0005, # Learning rate menos agresivo que el de por defecto, para un descenso de gradiente suave y controlado con un batch sizze de 4. 
    val_split=0.2,
    visualize=False,
    verbose=True,
)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 230MB/s]


Using device: cuda
Found 3230 image files and 3230 label files
Training on 2584 images, validating on 646 images
Epoch: 1, Batch: 1/646, Loss: 2.7264, Time: 4.39s
Epoch: 1, Batch: 11/646, Loss: 2.9448, Time: 13.37s
Epoch: 1, Batch: 21/646, Loss: 2.5542, Time: 13.80s
Epoch: 1, Batch: 31/646, Loss: 2.7335, Time: 14.34s
Epoch: 1, Batch: 41/646, Loss: 2.1280, Time: 14.52s
Epoch: 1, Batch: 51/646, Loss: 1.4788, Time: 14.99s
Epoch: 1, Batch: 61/646, Loss: 1.7900, Time: 15.41s
Epoch: 1, Batch: 71/646, Loss: 1.8648, Time: 15.98s
Epoch: 1, Batch: 81/646, Loss: 1.1080, Time: 16.77s
Epoch: 1, Batch: 91/646, Loss: 3.2615, Time: 16.06s
Epoch: 1, Batch: 101/646, Loss: 1.1759, Time: 15.93s
Epoch: 1, Batch: 111/646, Loss: 1.0936, Time: 15.74s
Epoch: 1, Batch: 121/646, Loss: 1.0007, Time: 15.96s
Epoch: 1, Batch: 131/646, Loss: 1.1177, Time: 16.04s
Epoch: 1, Batch: 141/646, Loss: 1.0449, Time: 16.20s
Epoch: 1, Batch: 151/646, Loss: 0.8545, Time: 16.29s
Epoch: 1, Batch: 161/646, Loss: 0.7711, Time: 16.28

In [5]:
modelo_entrenado = f"/kaggle/working/best_model.pth"
ruta_prediccion = "/kaggle/working/prediccion_cepas.tif"


In [6]:
import os

print("Iniciando escaneo absoluto del directorio de trabajo...")
archivos_encontrados = 0

# Buscamos en todo /kaggle/working sin importar cuántas subcarpetas haya
for raiz, directorios, archivos in os.walk('/kaggle/working'):
    for archivo in archivos:
        if archivo.endswith(('.pth', '.pt', '.weights')):
            ruta_completa = os.path.join(raiz, archivo)
            peso_mb = os.path.getsize(ruta_completa) / (1024 * 1024)
            print(f"📍 ¡LOCALIZADO! -> {ruta_completa} ({peso_mb:.2f} MB)")
            archivos_encontrados += 1

if archivos_encontrados == 0:
    print("❌ Negativo. La librería no ha guardado nada tras 1 época.")

Iniciando escaneo absoluto del directorio de trabajo...
📍 ¡LOCALIZADO! -> /kaggle/working/maskrcnn/best_model.pth (168.09 MB)
📍 ¡LOCALIZADO! -> /kaggle/working/maskrcnn/final_model.pth (168.09 MB)
📍 ¡LOCALIZADO! -> /kaggle/working/maskrcnn/training_history.pth (0.00 MB)
